# SmolVLA × LIBERO-plus Spatial Fine-tuning ─ 最終課題教材 (Advanced)

## この教材の位置付け

**Basic (`smolvla_libero_plus_spatial_basic.ipynb`) を先に完走してから**取り組む Advanced 版です。

Advanced では:

1. LeRobot 公式実装 (`SmolVLAPolicy`) を使って **事前学習済み SmolVLA を LoRA で追加学習**
2. LIBERO-plus の **4 suite × 各 3 タスク × 1 episode = 12 rollout / モデル** で広域評価
3. Section 8.0 のハイパラを工夫して LeRobot 版の性能を上げるのが挑戦ポイント

#### <font color="red">コントリビューション賞を目指す方は、指定のSlackへの工夫点の投稿をお願いします</font>


---

## Basic との関係

- **Basic**: PyTorch で自作 SmolVLA を実装 → 学習 → 評価 → `submission.json` を提出
- **Advanced (このノートブック)**: LeRobot 公式版で warm-start 学習 → 4 suite の広域評価

### Basic と Advanced の主な違い

- **主目的**: Basic は SmolVLA 内部の理解、Advanced は公式実装で実践
- **モデル**: Basic は自作の教育用簡略モデル、Advanced は LeRobot 公式 `SmolVLAPolicy`
- **Action head の初期値**: Basic は random、Advanced は事前学習済み重みから warm-start
- **学習方法**: Basic は PyTorch 自作ループで直接更新、Advanced は `lerobot-train` で LoRA 追加学習
- **評価範囲**: Basic は Spatial 2 タスク × 1 ep、Advanced は 4 suite × 3 タスク × 1 ep

### 共有される Drive キャッシュ

Basic が Section 5 で作った **LIBERO-plus assets の tar.gz キャッシュ**
(`Drive/MyDrive/smolvla_task/libero_plus_assets.tar.gz`) を、
Advanced 側も自動的に検出して再利用します。Basic を完走していれば、Advanced の Section 5
は tar.gz 展開だけで済むので短時間で終わります。

---

## 実行環境 (Colab)

Basic と同じ。ランタイムは **GPU (T4 以上)**、Drive マウントを承認。

## 使うデータと環境

- **学習データ**: `lerobot/libero_plus` (Basic と同じ)
- **評価環境**: `LiberoEnv` (LIBERO-plus 摂動シム)
- **評価範囲**: LIBERO Spatial + Object + Goal + Long-horizon の **4 suite / 各 3 タスク / 1 episode = 12 rollout / モデル**


## 📖 目次と各セクションの役割

### 【準備フェーズ】環境構築とデータ準備 (Basic と共通)

1. Colab ランタイム確認 — GPU の有無 / Python バージョンを確認、`WORKDIR` を設定
2. システムパッケージ — apt で `ffmpeg` / `libgl` 等を導入
3. LeRobot インストール — v0.6.0 を clone + editable install
4. HF 取得ヘルパ — 429 リトライ + キャッシュ優先の取得関数を定義
5. LIBERO-plus 環境準備 — **Basic の Drive tar.gz キャッシュを再利用** (未実行ならフル DL)
6. 学習・評価条件を設定 — データセット / seed / パス等の共通定数
7. Spatial 学習データ選抜 — 10 タスク × 5 episode = 50 episode を等間隔選抜

### 【★中核】LeRobot 版で学習 → 広域評価

**Section 8 (LeRobot 版で学習/評価)** — 事前学習済み SmolVLA を LoRA で追加学習。 Section 9 に進む場合は Section 8 全体が必須。

- **8.0** Advanced 用ハイパラ (STEPS / LORA_R / LEARNING_RATE 等を工夫する起点)
- **8.1** 事前学習済み SmolVLA を DL (`lerobot/smolvla_libero_plus`)
- **8.2** LeRobot subprocess で LoRA 学習 (`lerobot-train`)
- **8.3** LoRA をベースへマージ (`PeftModel.merge_and_unload`)
- **8.4** 対照ベースラインを準備 (追加学習前の重みを同 processor で保存)
- **8.5** 追加学習前後を Spatial で評価 (10 task × 1 ep × 2 モデル = 20 rollout)
- **8.6** LeRobot 版の成功率比較 (per-task Δ + CSV)
- **8.7** マージ済みモデルを zip 化 (提出用)
- **8.8** rollout 動画 optional (mp4 出力)

**Section 9 (アドバンスド評価)** — LIBERO-plus 4 suite × 3 task × 1 ep = 12 rollout / モデル (計 24)


## 1. Colabランタイムを確認する

ColabのランタイムをGPUへ変更してから実行してください。

In [ ]:
# ==============================================================
# 環境変数と実行前チェック + Google Drive マウント
# ==============================================================
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

# --- Google Drive をマウント (成果物の永続化) --------------------
# Colab の /content/ 配下はランタイム終了で消えるので、submission.json や
# 学習済モデルは Drive にも保存できるようにマウントしておく.
# 認証ポップアップが出るのでブラウザで許可すること (初回のみ).
from google.colab import drive

drive.mount("/content/drive")

# Drive 上に成果物をコピーする場所. `MyDrive/smolvla_task/` 配下.
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/smolvla_task")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"DRIVE_BACKUP_DIR: {DRIVE_BACKUP_DIR}")

# --- 作業ディレクトリ ------------------------------------------
# 実行速度重視で /content/ (ephemeral) に置く. Drive 直で作業すると
# 45 万小ファイルの展開などが遅くなる. 最終成果物だけ Drive にコピーする方針.
WORKDIR = Path("/content/workdir")
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"WORKDIR         : {WORKDIR}")

# --- Hugging Face Hub のノイズ抑制 ------------------------------
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
# HF / LeRobot のキャッシュを WORKDIR 配下へ固定
os.environ["HF_HOME"] = str(WORKDIR / "hf_cache")
os.environ["HF_LEROBOT_HOME"] = str(WORKDIR / "lerobot_cache")
# 長時間学習で起こりやすい断片化 OOM を回避
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# LeRobot v0.6.0 は Python 3.12 以上が必須
if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上が必要です。")

# GPU 必須
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPUランタイムを選択してください "
        "(Colab メニュー: ランタイム → ランタイムのタイプを変更 → GPU)"
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. システムパッケージを準備する

LeRobot、動画デコード、MuJoCoで必要になるパッケージを導入します。

In [ ]:
# ==============================================================
# サブプロセス実行ヘルパ + apt でシステムパッケージ導入
# ==============================================================


# 外部コマンドを stdout/stderr をまとめて捕捉しつつ実行する共通ヘルパ:
#   - 成功時はログを出さず notebook を汚さない
#   - 失敗時は末尾 6000 文字だけを例外に載せて可視化する
#   - stderr は stdout に統合して前後関係を保つ
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


# 長時間コマンド用: 1 行受信するたびに print する streaming 版。
# 進捗が見えない = 無限ループに見える問題を避けるためのヘルパ。
# - prefix を付けて何のコマンドの出力か分かるようにする
# - 直近ログ 200 行をリングバッファで保持し、失敗時ダンプに使う
def run_streaming(
    command: list[str],
    *,
    prefix: str = "",
    check: bool = True,
) -> int:
    from collections import deque

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    recent: deque[str] = deque(maxlen=200)
    assert process.stdout is not None
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        print(f"{prefix}{line}", flush=True)
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# 長時間コマンド用 (pip install 等の出力が数百行になるケース):
# 直近 tail 行だけを ipywidgets の HTML box で "上書き更新" 表示する.
# タイムライン全部を表示すると notebook が肥大化するので、末尾だけ見せる.
def run_streaming_rolling(
    command: list[str],
    *,
    prefix: str = "",
    tail: int = 10,
    check: bool = True,
) -> int:
    from collections import deque
    import ipywidgets as widgets
    from IPython.display import display

    recent: deque[str] = deque(maxlen=tail)
    _widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_widget)

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert process.stdout is not None
    _count = 0
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        _count += 1
        # 毎行更新すると UI が重いので 5 行ごとに widget を書換
        if _count % 5 == 0:
            _widget.value = (
                "<pre style='margin:0;font-size:11px;color:#666;"
                "line-height:1.3'>"
                + "\n".join(f"  {prefix}{l}" for l in recent)
                + "</pre>"
            )
    # 最終行まで反映
    _widget.value = (
        "<pre style='margin:0;font-size:11px;color:#666;line-height:1.3'>"
        + "\n".join(f"  {prefix}{l}" for l in recent)
        + "\n  ✅ done ({_count} lines total)".replace("{_count}", str(_count))
        + "</pre>"
    )
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# apt キャッシュ更新
run_quiet(["sudo", "apt-get", "update", "-qq"])
# 依存パッケージ内訳:
#   ffmpeg                : LeRobot の動画デコード
#   git / unzip           : このノートブックで clone / 展開に使用
#   libgl1 / libglib2.0-0 : OpenGL 系（MuJoCo/robosuite の描画）
#   libsm6 / libxext6     : X11 系ランタイム（描画ライブラリの依存）
#   libexpat1             : XML パーサ（MuJoCo の XML モデル読込）
#   libfontconfig1-dev    : matplotlib 用フォント設定
#   libmagickwand-dev     : Wand（ImageMagick binding）が LIBERO 経由で必要
run_quiet(
    [
        "sudo", "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "git",
        "unzip",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libexpat1",
        "libfontconfig1-dev",
        "libmagickwand-dev",
    ]
)

print("System packages ready.")

## 3. LeRobotをインストールする

LeRobot `v0.6.0`を使用します。
ColabでのLoRA学習に必要な互換性調整もこのセルで適用します。

In [ ]:
# ==============================================================
# LeRobot v0.6.0 の editable install と互換性パッチ
# ==============================================================
# 手順:
#   1. 既存 install の除去（torchao も同時に。SmolVLA 非対応のため）
#   2. 固定 tag の LeRobot を shallow clone（進捗を stream）
#   3. bf16 非対応 GPU なら SmolVLM を fp16 に書換
#   4. lerobot_train.py の冗長ログと tqdm を軽微パッチで抑制
#   5. editable install（training + smolvla + peft の extras）— 出力を stream
#   6. torchao を再度アンインストール（依存で復活するケースの保険）
#   7. sys.modules と sys.path を掃除して clone した src を優先させる
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = WORKDIR / "lerobot"
LEROBOT_SRC = LEROBOT_DIR / "src"

# --- 0. 安全ガード: torch のバージョン整合性チェック ------------
# 前回のセッションで torch を import 済みで、その後 pip install が
# torch を更新すると、C 拡張(.so)は旧版・Python 側は新版という
# 不整合状態になり "cannot import name '_EvalFrameOverride'" 等が発生する。
# 事前にディスク上と in-memory の torch バージョンを照合し、
# 差があればカーネル再起動を促す。
# torch.__version__ / importlib.metadata.version("torch") は環境によって
# '2.11.0+cu128' のように CUDA タグ (local version identifier) が付いたり
# 付かなかったりする. 両側から '+' 以降を落とした PEP 440 の public version だけで比較.
def _strip_local_version(v: str) -> str:
    return v.split("+", 1)[0]

_torch_version_in_memory = _strip_local_version(torch.__version__)
try:
    _torch_dist_version = _strip_local_version(
        importlib.metadata.version("torch")
    )
except importlib.metadata.PackageNotFoundError:
    _torch_dist_version = None

if _torch_dist_version and _torch_dist_version != _torch_version_in_memory:
    raise RuntimeError(
        f"torch mismatch: in-memory={_torch_version_in_memory} "
        f"on-disk={_torch_dist_version}. "
        "カーネルを再起動してこのセルを最初から実行し直してください。"
    )

# --- 1. 既存 install を除去（再実行安全性のため） ----------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

# --- 2. LeRobot を shallow clone（進捗を stream 表示） -----------
# ネットワークが遅いと数分かかるが、出力が多いので直近 10 行だけ表示
print("Cloning LeRobot...", flush=True)
run_streaming_rolling(
    [
        "git",
        "clone",
        "--progress",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ],
    prefix="[git] ",
    tail=10,
)

# --- 3. dtype 互換性パッチ -------------------------------------
# T4/V100 系など bf16 未対応 GPU では bf16 で load すると壊れるので fp16 に置換
smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

# --- 4. lerobot_train.py の軽微パッチ ---------------------------
# 学習 subprocess のログを絞る:
#   - 設定 pformat の INFO を DEBUG に落とす
#   - inside_slurm() 判定に依らず tqdm を強制 disable
train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

# --- 5. editable install（extras 込み） ------------------------
# --upgrade を付けない: torch を含む依存が既に満たされていれば
# 何も触らせない（在庫の C 拡張との不整合を避けるため）。
# 依存解決とビルドで 5〜20 分かかる. pip の出力は数百行になるので rolling 表示 (直近 10 行だけ).
print("Installing LeRobot (this may take a while)...", flush=True)
run_streaming_rolling(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ],
    prefix="[pip] ",
    tail=10,
)

# --- 5.5. インストール後 torch が動いていないことを再確認 --------
_torch_dist_version_after = _strip_local_version(
    importlib.metadata.version("torch")
)
if _torch_dist_version_after != _torch_version_in_memory:
    raise RuntimeError(
        f"pip install が torch を {_torch_version_in_memory} → "
        f"{_torch_dist_version_after} に変更しました。"
        "カーネルを再起動してこのセルを最初から実行し直してください。"
    )

# --- 6. torchao 再アンインストール ------------------------------
# 上の install 中に依存として復活することがあるため保険としてもう一度消す
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

# --- 7. sys.modules と sys.path の掃除 --------------------------
# 前回セル実行時に読まれた古い lerobot/torchao モジュールを追い出す
for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

# clone した src を最優先で解決させるため sys.path を差し替える
sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

# torchao が完全に消えたかチェック。残っていると SmolVLA が起動時に落ちる
try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchaoの削除に失敗しました。")

# 実際に import して、read 先が今 clone した src 配下であることを保証
# 初回 import は torch/CUDA 初期化で 30〜60 秒かかる場合がある
print("Importing lerobot (torch init may take up to a minute)...", flush=True)
import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobotの読込先が正しくありません。")

print("LeRobot ready.")

## 4. 公開ファイルの取得処理を用意する

キャッシュを優先し、匿名アクセスの制限時は自動的に再試行します。

In [ ]:
# ==============================================================
# Hugging Face 取得のリトライ + キャッシュ優先ラッパ
# ==============================================================
# 匿名アクセスは 429 (Too Many Requests) を食らいやすいので指数バックオフを
# 実装し、既存キャッシュがあれば実 DL を回避する。
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


# 429 のみ最大 6 回リトライ。それ以外のエラーは即再送出する。
def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            # 429 以外はリトライしない（404 などを叩き続けない）
            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            # サーバが Retry-After を返せば尊重、無ければ指数バックオフ
            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Faceからの取得に失敗しました。"
    ) from last_error


# まず local_files_only でキャッシュ確認、無ければ実 DL する。
# 大きなモデル/データセットを何度も再取得しないための最適化。
def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        # キャッシュヒット時はネットワークに触れず即返す
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        # キャッシュミス → 429 リトライ付きで実 DL
        # max_workers=1 でサーバに優しくアクセス（429 回避）
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 5. LIBERO-plus 評価環境を準備する

MuJoCo / robosuite / LIBERO-plus fork をインストールし、評価用アセットを展開します。
このステップは **PyTorch 版評価 (section 10)** と **LeRobot 版評価 (section 8)** の両方で必要なので、
早めにやっておきます。

**assets.zip (約 6GB) のダウンロード + 展開があるため、このセルは時間がかかります。**

In [ ]:
# ==============================================================
# LIBERO-plus 評価環境をセットアップ
# ==============================================================
# 手順:
#   1. MuJoCo/robosuite 系の依存を pin 付きで install
#   2. LIBERO-plus fork を固定 SHA で clone/checkout
#   3. assets.zip を HF から取得して展開
#   4. ~/.libero/config.yaml を書いてアセット位置を教える
#   5. lerobot_eval.py に進捗表示用のパッチを当てる
#   6. sys.path / sys.modules を掃除して fork 側を優先ロード
from huggingface_hub import hf_hub_download
import time

LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_DIR = WORKDIR / "LIBERO-plus"
LIBERO_PLUS_PACKAGE_ROOT = (
    LIBERO_PLUS_DIR / "libero" / "libero"
)
# assets (45 万小ファイル 6GB) の保存先. /content/ 配下なので展開は高速.
LIBERO_PLUS_ASSETS_DIR = WORKDIR / "libero_plus_assets" / "assets"

# Colab のヘッドレス GPU 環境では EGL バックエンドで MuJoCo を描画
os.environ["MUJOCO_GL"] = "egl"

# --- 1. 既存 install を除去（再実行安全性） --------------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "hf-libero",
        "libero",
        "robosuite",
    ],
    check=False,
)

# --- 2. LIBERO-plus 依存の pin 付き install --------------------
# バージョンを揃えないと robosuite / mujoco の API 差でクラッシュするため厳格に固定
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "robosuite==1.4.1",  # LIBERO-plus が期待する固定版
        "bddl==1.0.1",
        "easydict==1.13",
        "mujoco==3.7.0",
        "matplotlib==3.10.8",
        "Wand==0.6.13",
        "scikit-image==0.25.2",
        "gym==0.26.2",
        "future",  # bddl 1.0.1 が依存宣言していない隠れ依存
    ]
)

# robosuite が期待どおりの版でロードされることを担保
if (
    importlib.metadata.version("robosuite")
    != "1.4.1"
):
    raise RuntimeError(
        "robosuite 1.4.1 is required."
    )

# --- 3. LIBERO-plus fork を clone -----------------------------
if not (LIBERO_PLUS_DIR / ".git").is_dir():
    shutil.rmtree(
        LIBERO_PLUS_DIR,
        ignore_errors=True,
    )
    run_quiet(
        [
            "git",
            "clone",
            "--quiet",
            "https://github.com/sylvestf/LIBERO-plus.git",
            str(LIBERO_PLUS_DIR),
        ]
    )

# 固定 SHA へ checkout。既存 clone に対しても再実行安全になるよう、
# checkout 失敗時のみ fetch でリカバリする
checkout = run_quiet(
    [
        "git",
        "-C",
        str(LIBERO_PLUS_DIR),
        "checkout",
        "--quiet",
        LIBERO_PLUS_SHA,
    ],
    check=False,
)

if checkout.returncode != 0:
    # SHA がローカルに無い（shallow clone 等）なら fetch してから checkout
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "fetch",
            "--quiet",
            "--depth",
            "1",
            "origin",
            LIBERO_PLUS_SHA,
        ]
    )
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "checkout",
            "--quiet",
            LIBERO_PLUS_SHA,
        ]
    )

# fork を editable install。--no-deps で pin 済みの依存を上書きされないようにする
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "-e",
        str(LIBERO_PLUS_DIR),
    ]
)

# --- 4. assets.zip を HF から DL・展開 -------------------------
# 大きな asset は fork 側に含まれないので Sylvest/LIBERO-plus から取得
if not LIBERO_PLUS_ASSETS_DIR.is_dir():
    assets_root = WORKDIR / "libero_plus_assets"
    assets_root.mkdir(parents=True, exist_ok=True)

    # ---- Drive キャッシュ戦略 ----
    # 優先度: (1) tar.gz キャッシュ (最速)  > (2) zip キャッシュ  > (3) HF DL
    # tar.gz を Drive に持っておくと、次回セッションから展開まで含めて数分で終わる.
    _drive_tar = DRIVE_BACKUP_DIR / "libero_plus_assets.tar.gz"
    _drive_zip = DRIVE_BACKUP_DIR / "libero_plus_assets.zip"

    if _drive_tar.is_file():
        # ---- 最速パス: tar.gz を Drive から取ってきて展開 ----
        print(
            f"Drive の tar.gz キャッシュから復元中 (unzip 不要): {_drive_tar}",
            flush=True,
        )
        _t0 = time.time()
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar

        # tar.gz は zip と違って中央ディレクトリを持たないため、
        # 通常はストリーム走査しないとファイル総数が分からない.
        # そこで前回の tar 作成時に .count サイドカーへ件数を書き出しておき、
        # あればそれを使って tqdm の total を即座に埋める.
        _drive_tar_count = _drive_tar.with_suffix(_drive_tar.suffix + ".count")
        _tar_total = None
        if _drive_tar_count.is_file():
            try:
                _tar_total = int(_drive_tar_count.read_text().strip())
                print(f"サイドカーから件数を取得: {_tar_total} files", flush=True)
            except Exception:
                _tar_total = None
        if _tar_total is None:
            # サイドカー無し (旧キャッシュ): getmembers() で 1 度走査する.
            # data は展開しないのでフル展開よりは速いが、それでも数十秒待つ.
            print("件数を数えています (初回のみ、少し時間かかる)...", flush=True)
            with tarfile.open(_drive_tar, "r:gz") as _tf:
                _tar_total = sum(1 for _ in _tf)

        with tarfile.open(_drive_tar, "r:gz") as _tf:
            _pbar_tx = _tqdm_tar(total=_tar_total, desc="tar 展開", unit="files")
            while True:
                _m = _tf.next()
                if _m is None:
                    break
                _tf.extract(_m, assets_root)
                _pbar_tx.update(1)
            _pbar_tx.close()
        print(f"tar 展開完了 (in {time.time()-_t0:.0f}s)", flush=True)

        # 古い zip キャッシュを削除 (Drive 容量節約)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"古い zip キャッシュを削除: {_drive_zip}", flush=True)
            except Exception:
                pass
        # → LIBERO_PLUS_ASSETS_DIR が存在するはず. 以下 DL/unzip はスキップ.
        _skip_dl_unzip = True
    else:
        _skip_dl_unzip = False

if not LIBERO_PLUS_ASSETS_DIR.is_dir() and not _skip_dl_unzip:
    # ---- 中速パス: zip キャッシュから復元 or HF DL ----
    local_zip = assets_root / "assets.zip"

    if _drive_zip.is_file() and not local_zip.is_file():
        print(f"Drive キャッシュから assets.zip を復元: {_drive_zip}", flush=True)
        shutil.copy2(_drive_zip, local_zip)

    if local_zip.is_file():
        archive_path = local_zip
        print(f"既存の assets.zip を使用: {archive_path}", flush=True)
    else:
        print("HF から assets.zip をダウンロード中 (6GB, 時間かかる)...", flush=True)
        # ---- HF DL 中は progress bar を有効化 ----
        # env-var の pop は huggingface_hub が import 時に disabled をキャッシュするため効かない.
        # 公式 API enable_progress_bars() / disable_progress_bars() で明示的に切り替える.
        from huggingface_hub.utils import (
            enable_progress_bars,
            disable_progress_bars,
        )
        enable_progress_bars()
        try:
            _t0 = time.time()
            archive_path = Path(
                run_hf_with_retry(
                    lambda: hf_hub_download(
                        repo_id="Sylvest/LIBERO-plus",
                        repo_type="dataset",
                        filename="assets.zip",
                        local_dir=assets_root,
                        token=False,
                    )
                )
            )
            _dl_sec = time.time() - _t0
        finally:
            disable_progress_bars()

        _size_mb = archive_path.stat().st_size / 1e6
        print(
            f"HF DL 完了: {_size_mb:.0f} MB in {_dl_sec:.0f}s "
            f"({_size_mb / max(_dl_sec, 1):.1f} MB/s)",
            flush=True,
        )
        # 注: この段階では Drive に zip を書かない.
        # 展開後に tar.gz を作って Drive に置く (下の方の処理) 方が
        # 次セッションからの復元も速くて経済的.
    extract_dir = assets_root / "extract"

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    # 展開は Python の zipfile + tqdm progress bar で行う.
    # ・全体進捗は tqdm バーで見せる (ETA / files/sec 含む).
    # ・直近 10 ファイル名を ipywidgets の HTML box で "上書き更新" する
    #   (subprocess の unzip -o だと 45 万行の出力になって notebook が固まるため).
    import zipfile
    import ipywidgets as widgets
    from IPython.display import display
    from tqdm.auto import tqdm
    from collections import deque

    print(f"Extracting {archive_path.name} to {extract_dir}...", flush=True)

    # 直近展開ファイル 10 件を表示するウィジェット
    _recent_paths: deque[str] = deque(maxlen=10)
    _recent_widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_recent_widget)

    _t0 = time.time()
    with zipfile.ZipFile(archive_path) as _zf:
        _members = _zf.infolist()
        _pbar = tqdm(_members, desc="unzip", unit="files")
        for _i, _m in enumerate(_pbar):
            _recent_paths.append(_m.filename)
            _zf.extract(_m, extract_dir)
            # 500 ファイルごとに widget を上書き (毎回だと重い)
            if _i % 500 == 0:
                _recent_widget.value = (
                    "<pre style='margin:0;font-size:11px;color:#666;"
                    "line-height:1.3'>"
                    + "\n".join(f"  {p}" for p in _recent_paths)
                    + "</pre>"
                )
        # 最終状態
        _recent_widget.value = (
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>"
            + f"  ✅ done: {len(_members)} files extracted"
            + "</pre>"
        )
    print(f"unzip 完了 (in {time.time()-_t0:.0f}s)", flush=True)

    # zip 内のディレクトリ階層は変わりうるので rglob で assets/ を探す。
    # ネストが浅いものを優先（本物の assets ルートに近いと想定）
    candidates = sorted(
        [
            path
            for path in extract_dir.rglob("assets")
            if path.is_dir()
        ],
        key=lambda path: len(path.parts),
    )

    if not candidates:
        raise FileNotFoundError(
            "LIBERO-plus assets not found."
        )

    LIBERO_PLUS_ASSETS_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    # 展開した本物の assets を fork 内の期待位置へ移動
    shutil.move(
        str(candidates[0]),
        str(LIBERO_PLUS_ASSETS_DIR),
    )
    # assets の移動が終わったので extract テンポラリだけ削除。
    shutil.rmtree(extract_dir, ignore_errors=True)

    # ---- 次セッション用に tar.gz を Drive にキャッシュ ----
    # zip + unzip のフルパスは重いので、展開後の assets を tar.gz 化して Drive に保存.
    # 次回セッションでは tar 展開だけになり、数分で復元できる.
    try:
        _t0 = time.time()
        print(
            f"Drive に tar.gz キャッシュを作成中...",
            flush=True,
        )
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar_c

        # 作成時は事前にファイル数をカウントできるので tqdm の total を渡せる
        _entries = list(LIBERO_PLUS_ASSETS_DIR.rglob("*"))
        _total_entries = len(_entries) + 1  # +1: root dir 自身
        _pbar_tc = _tqdm_tar_c(total=_total_entries, desc="tar 作成", unit="files")

        def _tar_filter(tarinfo):
            _pbar_tc.update(1)
            return tarinfo

        with tarfile.open(_drive_tar, "w:gz", compresslevel=1) as _tf:
            _tf.add(LIBERO_PLUS_ASSETS_DIR, arcname="assets", filter=_tar_filter)
        _pbar_tc.close()
        # 次回復元時に tqdm の total を即座に出せるよう、件数サイドカーを保存.
        _drive_tar.with_suffix(_drive_tar.suffix + ".count").write_text(
            str(_total_entries) + "\n"
        )
        print(f"tar.gz キャッシュ完了 (in {time.time()-_t0:.0f}s)", flush=True)
        # 古い zip キャッシュを削除 (Drive 容量節約)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"旧 zip キャッシュを削除: {_drive_zip}", flush=True)
            except Exception:
                pass
    except Exception as _e:
        print(f"tar.gz キャッシュ作成失敗 (無視して継続): {_e}", flush=True)

# --- 5. ~/.libero/config.yaml を書く --------------------------
# LIBERO は起動時にこのファイルを読んで assets/bddl/datasets/init_files の
# 場所を解決する。fork 側のパスに向ける
# ---- LIBERO の期待する assets パスへの symlink を作る ----
# LIBERO の robosuite Arena XML 読込は fork 内の相対パス
# ({LIBERO_PLUS_DIR}/libero/libero/assets/...) をハードコードで参照する.
# 一方で本 notebook は 45 万小ファイルの実体を LIBERO_PLUS_ASSETS_DIR に置いてある.
# 両者を symlink で繋ぐことで、LIBERO は "期待の場所" から実体を辿れる.
_libero_expected_assets = LIBERO_PLUS_DIR / "libero" / "libero" / "assets"
if _libero_expected_assets.is_symlink() or _libero_expected_assets.exists():
    if _libero_expected_assets.is_symlink():
        _libero_expected_assets.unlink()
    else:
        shutil.rmtree(_libero_expected_assets, ignore_errors=True)
_libero_expected_assets.parent.mkdir(parents=True, exist_ok=True)
_libero_expected_assets.symlink_to(LIBERO_PLUS_ASSETS_DIR)
print(f"symlink: {_libero_expected_assets} -> {LIBERO_PLUS_ASSETS_DIR}")

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(
    parents=True,
    exist_ok=True,
)
(libero_config_dir / "config.yaml").write_text(
    "\n".join(
        [
            f"assets: {LIBERO_PLUS_ASSETS_DIR}",
            (
                "bddl_files: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}"
            ),
            (
                "datasets: "
                f"{LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}"
            ),
            (
                "init_states: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}"
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)

# --- 6. lerobot_eval.py に進捗表示パッチを当てる -----------------
# デフォルトの評価スクリプトは冗長な log と tqdm を吐くので抑制。
# 代わりに "EVAL_PROGRESS task=x/y episode=a/b" 形式を stdout に流し、
# こちらのノート側でパースして表示する。
eval_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_eval.py"
)
source = eval_script.read_text(encoding="utf-8")

# 冗長な設定 pformat を DEBUG に落とす
source = source.replace(
    "logging.info(pformat(asdict(cfg)))",
    "logging.debug(pformat(asdict(cfg)))",
    1,
)
# 録画時以外に rollout 動画を最大 10 本描画してしまう箇所を 0 に固定
source = source.replace(
    "max_episodes_rendered = 0 if cfg.eval.recording else 10",
    "max_episodes_rendered = 0",
    1,
)
# tqdm を強制無効化
source = source.replace(
    "disable=inside_slurm()",
    "disable=True",
)

# 現在の task/episode 進捗を保持するグローバル辞書をスクリプトへ注入
progress_state = (
    '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
)
if progress_state not in source:
    import_anchor = "from tqdm import trange\n"
    if import_anchor not in source:
        raise RuntimeError(
            "Evaluation progress import anchor not found."
        )
    source = source.replace(
        import_anchor,
        import_anchor + "\n" + progress_state + "\n",
        1,
    )

# task ループの先頭で _EVAL_PROGRESS を更新するコードを差し込む
task_loop_anchor = (
    "        for i, (task_group, task_id, env) "
    "in enumerate(tasks):\n"
)
task_loop_patch = (
    task_loop_anchor
    + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
    + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n'
)
if (
    '_EVAL_PROGRESS["task_index"] = i + 1'
    not in source
):
    if task_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation task-loop anchor not found."
        )
    source = source.replace(
        task_loop_anchor,
        task_loop_patch,
        1,
    )

# episode ループの各周で進捗 1 行を stdout に流す
episode_loop_anchor = "    for batch_ix in progbar:\n"
episode_progress_line = (
    '        print('
    'f"EVAL_PROGRESS '
    "task={_EVAL_PROGRESS['task_index']}/"
    "{_EVAL_PROGRESS['task_total']} "
    'episode={batch_ix + 1}/{n_batches}", '
    "flush=True)\n"
)
if "EVAL_PROGRESS task=" not in source:
    if episode_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation episode-loop anchor not found."
        )
    source = source.replace(
        episode_loop_anchor,
        episode_loop_anchor + episode_progress_line,
        1,
    )

eval_script.write_text(
    source,
    encoding="utf-8",
)

# --- 7. sys.path 掃除と fork のロード検証 ---------------------
# pip install した LIBERO-plus fork が絶対に手前に来るようにする
libero_plus_path = str(LIBERO_PLUS_DIR)
sys.path = [
    item
    for item in sys.path
    if item != libero_plus_path
]
sys.path.insert(0, libero_plus_path)

# 古い LIBERO/robosuite モジュールキャッシュを追い出す
for module_name in list(sys.modules):
    if (
        module_name == "libero"
        or module_name.startswith("libero.")
        or module_name == "robosuite"
        or module_name.startswith("robosuite.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

import libero
from libero.libero import benchmark

# libero パッケージの読込先が fork ディレクトリ配下であることを保証
search_paths = [
    Path(path).resolve()
    for path in getattr(libero, "__path__", [])
]

if not any(
    LIBERO_PLUS_DIR.resolve() in path.parents
    or path == LIBERO_PLUS_DIR.resolve()
    for path in search_paths
):
    raise RuntimeError(
        "LIBERO-plus fork was not loaded."
    )

# benchmark モジュールについても同様に確認
benchmark_path = Path(
    benchmark.__file__
).resolve()

if (
    LIBERO_PLUS_DIR.resolve()
    not in benchmark_path.parents
):
    raise RuntimeError(
        "LIBERO-plus benchmark was not loaded."
    )

print("LIBERO-plus ready.")

## 6. 学習・評価条件を設定する

このセルで定義するのは **両ルート共通のもの** だけ:

- モデル / データセットの repo と revision
- LIBERO-Spatial 10 タスクのリスト
- 学習に使う episode 数と seed
- 出力パス、混合精度

各ルート固有の学習ハイパラは以下:
- PyTorch 版 → `section 9.3` の `TRAIN_STEPS_PY` 等
- LeRobot 版 → `section 8.0` の `STEPS` 等

In [ ]:
# ==============================================================
# 学習・評価に使う定数とパスをまとめて定義
# ==============================================================

# --- モデル / データセットの固定 revision -----------------------
# revision を commit hash で固定して再現性を担保
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = (
    "7bb70aa5bc92b82c9239142775d3a173103567ff"
)

# SmolVLA の Vision-Language backbone（500M パラメータ版）
VLM_REPO = (
    "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
)

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = (
    "f3f49f426d75030177b18778374005bc12ccd588"
)

# --- LIBERO-Spatial の 10 タスク（学習・評価で共通） -------------
# データセット側の task 名と厳密一致させる必要がある
SPATIAL_TASK_NAMES = [
    "pick up the black bowl from table center and place it on the plate",
    "pick up the black bowl next to the cookie box and place it on the plate",
    "pick up the black bowl next to the plate and place it on the plate",
    "pick up the black bowl next to the ramekin and place it on the plate",
    "pick up the black bowl on the cookie box and place it on the plate",
    "pick up the black bowl on the ramekin and place it on the plate",
    "pick up the black bowl on the stove and place it on the plate",
    "pick up the black bowl on the wooden cabinet and place it on the plate",
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the black bowl between the plate and the ramekin and place it on the plate",
]

# --- 学習・評価データ選抜 --------------------------------------
TRAIN_EPISODES_PER_TASK = 5   # 10 tasks × 5 = 50 episodes 選抜
SEED = 42                     # 乱数 seed
EVAL_SEED = 2026              # 評価 seed

# --- 出力パス --------------------------------------------------
OUTPUT_DIR = WORKDIR / "outputs" / "smolvla_libero_plus_spatial_lora"
MERGED_MODEL_DIR = WORKDIR / "smolvla_libero_plus_spatial_lora_merged"
BASELINE_MODEL_DIR = WORKDIR / "smolvla_libero_plus_baseline"

BASE_EVAL_DIR = WORKDIR / "eval" / "base"
FINETUNED_EVAL_DIR = WORKDIR / "eval" / "spatial_lora"
COMPARISON_CSV_PATH = WORKDIR / "libero_spatial_comparison.csv"
MERGED_ZIP_PATH = WORKDIR / "smolvla_libero_plus_spatial_lora_merged.zip"

# --- 混合精度モード --------------------------------------------
# bf16 対応 GPU（A100/H100/L40/RTX40 系など）なら bf16、それ以外は fp16
MIXED_PRECISION = (
    "bf16"
    if torch.cuda.is_bf16_supported()
    else "fp16"
)

## 7. Spatial 学習データを選ぶ

10タスクから各5エピソードを等間隔に選択します。

In [ ]:
# ==============================================================
# LIBERO-Spatial 10 タスクから各 5 episode を等間隔選択
# ==============================================================
# 学習用サブセットを再現可能に構築する。
# データセット側の task 名は表記揺れがありうるので正規化キーで突合し、
# 各 task の episode index から等間隔に 5 個を間引く（合計 50 episode）。
import re
from collections import defaultdict

from lerobot.datasets.dataset_metadata import (
    LeRobotDatasetMetadata,
)


# 小文字化・記号除去・空白正規化で突合用キーを作る。
def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


# メタデータの tasks 列は str/list どちらもあり得るので単一 str に落とす。
def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value

    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass

    return str(value)


# episode 群から等間隔に count 個を選ぶ（先頭と末尾を必ず含む）。
def choose_evenly_spaced(
    episode_indices: list[int],
    count: int,
) -> list[int]:
    if len(episode_indices) < count:
        raise ValueError(
            f"Task には episode が {len(episode_indices)} 件しかないため、"
            f"要求数 {count} を満たせません。"
        )
    positions = [
        round(
            index
            * (len(episode_indices) - 1)
            / (count - 1)
        )
        for index in range(count)
    ]

    return [
        episode_indices[position]
        for position in positions
    ]


# データセットのメタデータを取得（429 リトライ付き）
dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(
        DATASET_REPO,
        revision=DATASET_REVISION,
    )
)

# 「task 名 → その task に属する episode index リスト」を構築
task_to_episodes: dict[str, list[int]] = defaultdict(list)

for episode_index, task_cell in enumerate(
    dataset_metadata.episodes["tasks"]
):
    task_to_episodes[
        task_name_from_cell(task_cell)
    ].append(int(episode_index))

# 正規化キーから実際のタスク名（表記揺れ含む）を引くための辞書
available_by_normalized = {
    normalize_task_name(task_name): task_name
    for task_name in task_to_episodes
}

# Spatial の 10 タスク各々から 5 episode を選ぶ
selected_by_task: dict[str, list[int]] = {}

for task_name in SPATIAL_TASK_NAMES:
    actual_task = available_by_normalized.get(
        normalize_task_name(task_name)
    )

    if actual_task is None:
        raise RuntimeError(
            f"Spatial task not found: {task_name}"
        )

    selected_by_task[actual_task] = choose_evenly_spaced(
        task_to_episodes[actual_task],
        TRAIN_EPISODES_PER_TASK,
    )

# フラットな episode index のソート済みリストへ集約
EPISODE_INDICES = sorted(
    episode_index
    for episode_indices in selected_by_task.values()
    for episode_index in episode_indices
)

# 期待: 10 tasks × 5 = 50 個
if len(EPISODE_INDICES) != 50:
    raise RuntimeError("Episode selection failed.")

print("Training data: 10 tasks × 5 episodes = 50 episodes")

## 8. LeRobot 版で LoRA 追加学習

> **📌 Section 9 に進む場合は必須**
> Section 9 のアドバンスド評価は、Section 8.3 (merge 済みモデル) と 8.4 (対照ベースライン)
> で作ったディレクトリを入力に取ります。Section 9 だけを実行する構成にはできないので、
> Section 8 全体を先に完走してから Section 9 に進んでください。


> **📎 Section 9 の入力を作る役割**
> ここで作った学習済みモデル (`MERGED_MODEL_DIR`) と対照ベースライン (`BASELINE_MODEL_DIR`)
> が Section 9 のアドバンスド評価 (4 suite × 各 3 タスク × 1 ep) の入力になります。
> `SmolVLAPolicy.save_pretrained` を経由する限り、Section 9 に渡す形式は
> 自動で正しく整うため、通常は Section 6 のハイパラや Section 8.2 の
> `lerobot-train` 引数を工夫するだけで、Section 9 の評価コードは触らなくて OK。
> 詳細は **Section 9 冒頭の「入力形式契約」** 参照。

> **📌 LoRA の本領はここで発揮される**
> Basic ノートブックでは Action head を from-scratch で学習したので LoRA を使いませんでした
> (random 初期値に低ランク差分を足しても LoRA の利点が薄いため)。
> Advanced では `lerobot/smolvla_libero_plus` の **事前学習済み Action head** を
> warm-start してから追加学習するので、LoRA が「事前学習を保ったまま少量差分だけ学習」
> という本来の使い方になります (`--peft.method_type=LORA --peft.r=16`)。

Basic ノートブックでは PyTorch で SmolVLA を **from-scratch** から書き下しました。
一方 LeRobot には `SmolVLAPolicy` (`lerobot/smolvla_libero_plus` の事前学習済み重み) と
`lerobot-train` / `lerobot-eval` の CLI がすでに用意されています。

これを使うと:

- **事前学習済みの action head** を warm-start できる (最初から高い性能で始まる)
- 学習は subprocess で走るのでノートブック側は待つだけ
- 評価も CLI 経由で LIBERO Spatial 10 タスク × 数エピソードを回せる

Section 8.1〜8.6 で追加学習 → Spatial での比較評価まで行い、
Section 8.7〜8.8 で成果物 (zip / 動画) を作ります。
そのあと Section 9 で 4 suite への広域評価に進みます。


### 8.0 LeRobot 版のハイパラ

LeRobot subprocess (`lerobot-train` / `lerobot-eval`) で使うハイパラをここで定義します。

---

### 🛠️ Section 8 で工夫できる箇所

以下の変更は **`SmolVLAPolicy.save_pretrained` の出力形式を保つ**ので、
Section 9 のアドバンスド評価コードを **触らずにそのまま** 効果を測れます:

- **LoRA rank / alpha** — 8.0 の `LORA_R`, `LORA_ALPHA` を変える。表現力と学習パラメタ数のトレードオフ。
- **学習 step 数** — 8.0 の `STEPS` を変える。収束具合に効く。
- **学習率スケジュール** — 8.0 の `LEARNING_RATE`, `FINAL_LEARNING_RATE`, `WARMUP_STEPS`。学習の安定性。
- **バッチサイズ** — 8.0 の `BATCH_SIZE`。学習効率 (VRAM に注意)。
- **seed 変更** — Section 6 の `SEED` (共通)。学習の初期条件。
- **LoRA 対象モジュール** — Section 8.2 の `lerobot-train` 引数に `--peft.target_modules` を追加。どの層を LoRA するか。
- **学習エピソード選抜** — Section 7 の `EPISODE_INDICES` を変える。どのデータで学習するか。

---

### ⚠️ 契約を破りやすい変更 (Advanced 課題を狙う人向け注意)

以下は **Section 9 の評価が動かなくなる可能性がある**変更。試すなら影響を理解した上で:

- **`BASE_MODEL_REPO` を SmolVLA 以外に変更 (例: pi0)** — Section 8.3 の merge が SmolVLAPolicy 決め打ちなので type mismatch。 → 回避策: Section 8.3, 8.4 も対応 policy class に書き換える。
- **Custom な nn.Module を追加** — `save_pretrained` が拾わず保存されない。 → 回避策: LeRobot の `PreTrainedPolicy` を継承した custom class を作り、config に登録する。
- **`MERGED_MODEL_DIR` を手作りで別形式に** — ファイル欠損で `lerobot-eval` がロード失敗。 → 回避策: `SmolVLAPolicy.save_pretrained` を必ず経由する。
- **`train_expert_only=False` に変更** — VLM 側も学習対象になり時間・メモリ増大、収束不安定。 → 回避策: 慎重に。default (True) 推奨。
- **`chunk_size` 変更** — eval も追従するが LIBERO env との fps 整合を保つこと。 → 回避策: 8.0 に `chunk_size` は無いので `--policy.chunk_size` を 8.2 引数に足す必要あり。
- **画像リサイズ変更** — eval env の camera 出力と食い違う。 → 回避策: Section 9 の `--env.observation_height/width` も揃える。

以上を踏まえた上で、**まずは 8.0 のハイパラ書き換えから始めるのが安全**です。


In [ ]:
# ==============================================================
# 8.0 LeRobot 版のハイパラ
# ==============================================================
# LeRobot subprocess (lerobot-train / lerobot-eval) で使うハイパラ.

# ---- LoRA 追加学習用 (section 8.2 の lerobot-train が使う) ----
STEPS               = 3000        # 学習総 step 数
LOG_FREQ            = 100         # loss print 頻度
BATCH_SIZE          = 1           # ミニバッチサイズ (VRAM に応じて調整)
LEARNING_RATE       = 3e-4        # peak learning rate
FINAL_LEARNING_RATE = 3e-5        # cosine decay 後の lr
WARMUP_STEPS        = 100         # linear warmup step 数
LORA_R              = 16          # LoRA 低ランク次元 (4/8/16/32 が定番)
LORA_ALPHA          = 16          # LoRA スケーリング (通常 = r or 2r)

# ---- LeRobot 評価用 (section 8.5 の lerobot-eval が使う) ----
EVAL_TASK_IDS          = list(range(10))   # LIBERO-Spatial の task_id (全 10)
# Section 8.5 は 10 task × 1 ep × 2 model = 20 rollout
EVAL_EPISODES_PER_TASK = 1

# 現在値をログ表示 (どのハイパラで学習/評価するかを確認)
print("[section 8.0] 使用するハイパラ:")
print(f"  learning: STEPS={STEPS}, LR={LEARNING_RATE}, WARMUP={WARMUP_STEPS},")
print(f"            BATCH={BATCH_SIZE}, SEED={SEED}")
print(f"  LoRA    : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  eval    : TASK_IDS={EVAL_TASK_IDS},")
print(f"            EPISODES_PER_TASK={EVAL_EPISODES_PER_TASK}, SEED={EVAL_SEED}")

### 8.1 事前学習済み SmolVLA を DL

In [ ]:
# ==============================================================
# 追加学習の初期重みをダウンロード
# ==============================================================
# 必要最小限のファイルのみ取得する。README や eval/**（動画ファイル）は落とさない。
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    allow_patterns=[
        "config.json",                        # モデル構成
        "model.safetensors",                  # 本体 weight
        "train_config.json",                  # 学習時設定（再利用のため）
        "policy_preprocessor.json",           # 入力前処理設定
        "policy_preprocessor*.safetensors",   # 入力側 stats
        "policy_postprocessor.json",          # 出力後処理設定
        "policy_postprocessor*.safetensors",  # 出力側 stats
    ],
    ignore_patterns=[
        "README.md",
        "eval/**",  # 公開 repo に含まれる評価動画（重い）は不要
    ],
)

# 本体 weight が確実に取れていることを確認
if not (
    BASE_MODEL_LOCAL / "model.safetensors"
).is_file():
    raise FileNotFoundError("Base model not found.")

print("Base model ready.")

### 8.2 LeRobot subprocess で LoRA 学習

100 stepごとに平均lossとlearning rateを表示します。

In [ ]:
# ==============================================================
# lerobot-train を subprocess で起動し進捗をリアルタイム表示
# ==============================================================
# lerobot-train の CLI をそのまま呼んで LoRA 学習を実行。
# stdout をパースして step/loss/lr を LOG_FREQ ごとに 1 行だけ表示する。
import re
from collections import deque

# EPISODE_INDICES を CLI 引数として渡すため JSON 配列文字列に整形
episodes_json = (
    "["
    + ",".join(map(str, EPISODE_INDICES))
    + "]"
)

# --- lerobot-train の CLI 引数 ---------------------------------
command = [
    "lerobot-train",
    # 初期重みと VLM backbone のパス
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.vlm_model_name={VLM_REPO}",
    # Hub への push は行わない
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    # 入出力 features はチェックポイント側の値を尊重
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    # LoRA 対象は expert 側のみ。VLM 側は凍結する
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    # 学習率と cosine スケジューラ
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    # データセットと選抜済み episode
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    # 出力ディレクトリ / ジョブ名
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=smolvla_libero_plus_spatial_lora",
    # 学習ステップ / batch / DataLoader
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=0",            # subprocess 内での追加 worker は無効化
    "--persistent_workers=false",
    # 学習中の env 評価は無効（このノート側で別途評価する）
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    # checkpoint は最終 step のみローカル保存
    "--save_checkpoint=true",
    f"--save_freq={STEPS}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    # LoRA 設定
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

# --- 子プロセスの環境変数 ---------------------------------------
training_env = os.environ.copy()
# clone した src を PYTHONPATH の先頭に入れてパッチ済み LeRobot を使わせる
training_env["PYTHONPATH"] = (
    str(LEROBOT_SRC)
    + os.pathsep
    + training_env.get("PYTHONPATH", "")
)
# accelerate に混合精度モード（bf16/fp16）を伝える
training_env["ACCELERATE_MIXED_PRECISION"] = (
    MIXED_PRECISION
)
# stdout を行バッファ化して進捗をリアルタイム表示できるようにする
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_HUB_VERBOSITY"] = "error"
training_env["TQDM_DISABLE"] = "1"
training_env["PYTHONWARNINGS"] = "ignore"

# 出力ディレクトリはクリーンな状態から始める
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print("Preparing data and starting training...")

# --- サブプロセスを起動して stdout を逐次読み --------------------
process = subprocess.Popen(
    command,
    cwd=LEROBOT_DIR,
    env=training_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# 直近ログ 80 行を保持しておき、失敗時にダンプする（診断用）
recent_lines: deque[str] = deque(maxlen=80)
report_step = LOG_FREQ

assert process.stdout is not None

# "step:... loss:... lr:..." 行だけを拾って整形表示する
for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()

    if not line:
        continue

    recent_lines.append(line)

    if "step:" in line and "loss:" in line:
        loss_match = re.search(r"loss:([0-9.eE+-]+)", line)
        lr_match = re.search(r"lr:([0-9.eE+-]+)", line)

        loss = loss_match.group(1) if loss_match else "n/a"
        lr = lr_match.group(1) if lr_match else "n/a"

        print(
            f"step {report_step:4d}/{STEPS}  "
            f"loss={loss}  lr={lr}"
        )
        report_step += LOG_FREQ

return_code = process.wait()

# 失敗時は末尾ログを吐いてから raise（原因調査用）
if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(
        f"Training failed: {return_code}"
    )

print("Training complete.")

### 8.3 LoRA をベースへマージ

LoRA差分を元weightへ統合し、通常のLeRobotモデルとして保存します。

In [ ]:
# ==============================================================
# LoRA アダプタをベース weight へマージして単体モデル化
# ==============================================================
# 学習後の adapter を PeftModel でロード → merge_and_unload で
# LoRA 差分をベース weight に足し込み、通常の LeRobot モデル形式で保存する。
# こうすることで評価時に peft を介さず素の推論経路で動作する。
import contextlib
import gc
import io
import json

from peft import PeftModel
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
)

# lerobot-train が保存した最終 checkpoint（adapter が入っている）
checkpoint_dir = (
    OUTPUT_DIR
    / "checkpoints"
    / f"{STEPS:06d}"
    / "pretrained_model"
)

if not (
    checkpoint_dir / "adapter_model.safetensors"
).is_file():
    raise FileNotFoundError("Final adapter not found.")

# GPU メモリを空けてから CPU 上でマージ（VRAM を節約）
gc.collect()
torch.cuda.empty_cache()

# --- ベース + LoRA のロード設定 ---------------------------------
merge_config = PreTrainedConfig.from_pretrained(
    checkpoint_dir
)
merge_config.device = "cpu"                     # マージは CPU で実行
merge_config.pretrained_path = BASE_MODEL_LOCAL # ベース weight の場所を明示
merge_config.use_peft = False                   # マージ後は peft 経路を使わない

# ロード時の noisy な info ログを黙らせる
quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    # 1) ベース SmolVLA を CPU にロード
    base_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=merge_config,
        strict=False,
    )

    # 2) 学習済み adapter を被せる
    peft_policy = PeftModel.from_pretrained(
        base_policy,
        checkpoint_dir,
        is_trainable=False,
        torch_device="cpu",
    )

    # 3) LoRA を base に足し込んで単体モデル化
    merged_policy = peft_policy.merge_and_unload(
        safe_merge=True
    )

# --- マージ済みモデルを保存 -------------------------------------
shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 保存 config は「単体モデル・再 push 無し・VLM は再ロードしない」に固定
merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name = VLM_REPO

merged_policy.save_pretrained(MERGED_MODEL_DIR)

# 学習中に更新された前処理/後処理 stats も同じディレクトリへコピーしないと
# 評価時に入出力スケーリングが崩れる
for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(
            source_path,
            MERGED_MODEL_DIR / source_path.name,
        )

# --- マージ結果に LoRA パラメータが残っていないか検証 ------------
merged_weights_path = (
    MERGED_MODEL_DIR / "model.safetensors"
)

with safe_open(
    merged_weights_path,
    framework="pt",
    device="cpu",
) as weights:
    if any(
        "lora_" in key.lower()
        for key in weights.keys()
    ):
        raise RuntimeError(
            "LoRA parameters remain after merge."
        )

# CPU 上に膨らんだモデルを片付ける（次の baseline ロードが重い）
del peft_policy
del base_policy
del merged_policy

gc.collect()
torch.cuda.empty_cache()

print("Merged model ready.")

### 8.4 対照ベースラインを準備

公開weightを追加学習モデルと同じ入力schema・processorへ揃えます。

In [ ]:
# ==============================================================
# 比較用ベースラインを追加学習モデルと同じ schema で保存し直す
# ==============================================================
# 公開の初期重みをそのまま評価するのではなく、マージ済みモデルと同じ
# processor 設定・入出力 features に揃えた「対照モデル」として保存し直す。
# これで評価時の前処理/後処理を完全に一致させ、比較を公正にする。

# マージ済みモデルの config を土台にする（入出力 schema が同一になる）
baseline_config = PreTrainedConfig.from_pretrained(
    MERGED_MODEL_DIR
)
baseline_config.device = "cpu"
baseline_config.pretrained_path = BASE_MODEL_LOCAL
baseline_config.use_peft = False
baseline_config.load_vlm_weights = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    # 公開の初期 weight を読みつつ、config だけマージ済み側と揃える
    baseline_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=baseline_config,
        strict=False,
    )

shutil.rmtree(BASELINE_MODEL_DIR, ignore_errors=True)
BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 保存 config を単体モデル運用向けに固定（マージ済みモデルと同じ扱い）
baseline_policy.config.use_peft = False
baseline_policy.config.pretrained_path = None
baseline_policy.config.push_to_hub = False
baseline_policy.config.repo_id = None
baseline_policy.config.device = None
baseline_policy.config.load_vlm_weights = False
baseline_policy.config.vlm_model_name = VLM_REPO
baseline_policy.save_pretrained(BASELINE_MODEL_DIR)

# processor stats はマージ済み側からコピー（両者を完全一致させる）
for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in MERGED_MODEL_DIR.glob(pattern):
        shutil.copy2(
            source_path,
            BASELINE_MODEL_DIR / source_path.name,
        )

del baseline_policy
gc.collect()
torch.cuda.empty_cache()

print("Baseline ready.")

### 8.5 LeRobot subprocess で追加学習前後を評価

追加学習前後の2モデルを、同じ10タスク・同じseedで評価します。
評価は1モデルにつき10 rollout、2モデル合計で20 rolloutです。

In [ ]:
# ==============================================================
# 追加学習前後の 2 モデルを同一条件で評価
# ==============================================================
# lerobot-eval を subprocess で 2 回実行し、Section 5 で仕込んだ
# "EVAL_PROGRESS ..." 行をパースしてリアルタイム進捗を表示する。
import json
import re
from collections import deque

# LIBERO 内部のカメラ名 → SmolVLA が期待する名前へのマッピング
EVAL_CAMERA_MAPPING = {
    "agentview_image": "front",           # 俯瞰カメラ
    "robot0_eye_in_hand_image": "wrist",  # ハンドカメラ
}


# lerobot-eval CLI 引数を組み立てる。ベース/マージ済みで共通の設定。
def build_eval_command(
    policy_path: Path,
    output_dir: Path,
) -> list[str]:
    return [
        "lerobot-eval",
        f"--policy.path={policy_path}",
        "--policy.device=cuda",
        "--policy.use_amp=false",
        "--env.type=libero",
        "--env.is_libero_plus=true",
        "--env.task=libero_spatial",
        (
            "--env.task_ids="
            + json.dumps(
                EVAL_TASK_IDS,
                separators=(",", ":"),
            )
        ),
        (
            "--env.camera_name_mapping="
            + json.dumps(
                EVAL_CAMERA_MAPPING,
                separators=(",", ":"),
            )
        ),
        # 観測画像は 256x256（SmolVLA の入力仕様に合わせる）
        "--env.observation_height=256",
        "--env.observation_width=256",
        # relative action 空間（LIBERO-plus のデフォルト）
        "--env.control_mode=relative",
        "--env.max_parallel_tasks=1",  # 逐次実行（VRAM 節約）
        "--eval.batch_size=1",
        (
            "--eval.n_episodes="
            f"{EVAL_EPISODES_PER_TASK}"
        ),
        "--eval.use_async_envs=false",
        "--eval.recording=false",
        f"--seed={EVAL_SEED}",
        f"--output_dir={output_dir}",
    ]


# 1 モデル分の評価を回して eval_info.json を dict で返す。
def run_evaluation(
    policy_path: Path,
    output_dir: Path,
    label: str,
) -> dict:
    shutil.rmtree(
        output_dir,
        ignore_errors=True,
    )

    # 評価用の子プロセスへ渡す環境変数
    eval_env = os.environ.copy()
    eval_env["MUJOCO_GL"] = "egl"
    # LIBERO-plus fork とパッチ済み LeRobot src の両方を優先パスに入れる
    eval_env["PYTHONPATH"] = (
        str(LIBERO_PLUS_DIR)
        + os.pathsep
        + str(LEROBOT_SRC)
        + os.pathsep
        + eval_env.get("PYTHONPATH", "")
    )
    eval_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_HUB_VERBOSITY"] = "error"
    eval_env["TQDM_DISABLE"] = "1"
    eval_env["PYTHONWARNINGS"] = "ignore"
    eval_env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        build_eval_command(policy_path, output_dir),
        cwd=LEROBOT_DIR,
        env=eval_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    # 失敗時ダンプ用の直近ログ
    recent_lines: deque[str] = deque(maxlen=120)

    # Section 5 で仕込んだ "EVAL_PROGRESS task=x/y episode=a/b" 行を拾う正規表現
    progress_pattern = re.compile(
        r"^EVAL_PROGRESS "
        r"task=(\d+)/(\d+) "
        r"episode=(\d+)/(\d+)$"
    )

    # tqdm で全 rollout の進捗を出す
    from tqdm.auto import tqdm
    _total_rollouts = (
        len(EVAL_TASK_IDS) * EVAL_EPISODES_PER_TASK
    )
    _pbar_eval = tqdm(total=_total_rollouts, desc=f"eval[{label}]", unit="ep")

    assert process.stdout is not None
    for raw_line in process.stdout:
        line = raw_line.replace("\r", "").strip()

        if not line:
            continue

        recent_lines.append(line)
        match = progress_pattern.match(line)

        if match:
            ti, tt, ei, et = match.groups()
            _pbar_eval.update(1)
            _pbar_eval.set_postfix_str(f"task {ti}/{tt} ep {ei}/{et}")

    _pbar_eval.close()
    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            "\n".join(recent_lines)
        )

    # 評価結果 JSON を読んで返す
    result_path = (
        output_dir
        / "eval_info.json"
    )

    if not result_path.is_file():
        raise FileNotFoundError(
            result_path
        )

    return json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


# 追加学習前の（processor を揃えた）ベースラインを評価
BASE_EVAL_INFO = run_evaluation(
    BASELINE_MODEL_DIR,
    BASE_EVAL_DIR,
    "Base model",
)

# LoRA マージ済みモデルを同じ seed・同じ tasks で評価
FINETUNED_EVAL_INFO = run_evaluation(
    MERGED_MODEL_DIR,
    FINETUNED_EVAL_DIR,
    "Spatial LoRA",
)

print("Evaluation complete.")

### 8.6 LeRobot 版の成功率比較

`Δ (pp)`は、追加学習後から追加学習前を引いた成功率差です。

In [ ]:
# ==============================================================
# タスク単位の成功率と Δ (pp) を表にまとめる
# ==============================================================
import pandas as pd
from IPython.display import display


# eval_info.json の per_task から task_id → 成功率(%) を作る。
def per_task_success(
    eval_info: dict,
) -> dict[int, float]:
    result: dict[int, float] = {}

    for task_info in eval_info["per_task"]:
        task_id = int(task_info["task_id"])
        successes = task_info["metrics"]["successes"]
        # bool リスト → True の割合を %
        result[task_id] = (
            100.0
            * sum(bool(value) for value in successes)
            / len(successes)
        )

    return result


base_per_task = per_task_success(BASE_EVAL_INFO)
finetuned_per_task = per_task_success(
    FINETUNED_EVAL_INFO
)

# --- 各 task 行を構築 -----------------------------------------
rows = []

for task_id in EVAL_TASK_IDS:
    base_score = base_per_task[task_id]
    finetuned_score = finetuned_per_task[task_id]

    rows.append(
        {
            "Task ID": task_id,
            "Task": SPATIAL_TASK_NAMES[task_id],
            "Base (%)": base_score,
            "Spatial LoRA (%)": finetuned_score,
            # Δ (pp) は「追加学習後 − 追加学習前」のパーセンテージポイント差
            "Δ (pp)": finetuned_score - base_score,
        }
    )

# --- Overall 行（LIBERO-Spatial 全体の pc_success） ------------
base_overall = float(
    BASE_EVAL_INFO["overall"]["pc_success"]
)
finetuned_overall = float(
    FINETUNED_EVAL_INFO["overall"]["pc_success"]
)

rows.append(
    {
        "Task ID": "Overall",
        "Task": "LIBERO-Spatial",
        "Base (%)": base_overall,
        "Spatial LoRA (%)": finetuned_overall,
        "Δ (pp)": finetuned_overall - base_overall,
    }
)

# DataFrame 化して CSV に落としつつ notebook に表示
comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(
    COMPARISON_CSV_PATH,
    index=False,
)

display(comparison_df.round(1))

print(
    f"Overall: {base_overall:.1f}% → "
    f"{finetuned_overall:.1f}% "
    f"({finetuned_overall - base_overall:+.1f} pp)"
)

### 8.7 マージ済みモデルを zip 化

In [ ]:
# ==============================================================
# マージ済みモデルを zip 化して取り出せる状態にする
# ==============================================================
# Colab では files.download でブラウザに zip を落とせる.
# ノートブック終了時に /content/ が消えるので、必ず外にコピーすること.
from zipfile import ZIP_STORED, ZipFile
from google.colab import files

# 既に zip があれば消す（再実行時のため）
if MERGED_ZIP_PATH.exists():
    MERGED_ZIP_PATH.unlink()

# ZIP_STORED（無圧縮）を選んでいるのは safetensors が既に圧縮済みで、
# 再圧縮しても縮まないため。IO 時間だけ増えるので無圧縮で良い。
with ZipFile(
    MERGED_ZIP_PATH,
    mode="w",
    compression=ZIP_STORED,
    allowZip64=True,  # safetensors が 4GB 超になるケースの保険
) as archive:
    for file_path in sorted(
        MERGED_MODEL_DIR.rglob("*")
    ):
        if file_path.is_file():
            # zip 内では MERGED_MODEL_DIR の basename を root にする
            archive.write(
                file_path,
                arcname=(
                    Path(MERGED_MODEL_DIR.name)
                    / file_path.relative_to(
                        MERGED_MODEL_DIR
                    )
                ),
            )

print(f"Saved: {MERGED_ZIP_PATH}")
files.download(str(MERGED_ZIP_PATH))
files.download(str(COMPARISON_CSV_PATH))

### 8.8 LeRobot 版の rollout 動画 (optional)

学習後のモデルが実際にどう動くかを mp4 で可視化します。
1 タスク × 数エピソードだけ `--eval.recording=true` で eval を実行し、
できた mp4 を notebook 上に埋め込みます。

`VIDEO_TASK_ID` と `VIDEO_N_EPISODES` を変えれば、他のタスクや本数も撮れます。

In [ ]:
# ==============================================================
# 動画セル: マージ済みモデルの rollout を mp4 で見る
# ==============================================================
# 手順:
#   1. lerobot_eval.py の録画数を LEROBOT_MAX_RECORDED 環境変数で
#      制御できるよう idempotent にパッチ (Section 5 のパッチを拡張)
#   2. 1 タスク × N 本の rollout を recording=true で実行
#   3. できた mp4 を notebook に埋め込み表示
from IPython.display import Video, display

VIDEO_TASK_ID = 0          # LIBERO-Spatial の task 0 を撮る (別の task なら 0〜9)
VIDEO_N_EPISODES = 2       # 撮る本数。mp4 が notebook に埋め込まれるので増やしすぎ注意
VIDEO_OUT_DIR = WORKDIR / "eval" / "video"

# --- 1. eval_script を env-var 対応にパッチ (idempotent) --------
# Section 5 で入れた "max_episodes_rendered = 0" (固定) を
# 環境変数 LEROBOT_MAX_RECORDED で切り替えられる形に置換する。
eval_script_path = (
    LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_eval.py"
)
_src = eval_script_path.read_text(encoding="utf-8")

# eval_script は module top で os を import していないため
# __import__("os") を使って self-contained な式にする
_flexible_patch = (
    'max_episodes_rendered = '
    'int(__import__("os").environ.get("LEROBOT_MAX_RECORDED", "0"))'
)
_broken_patch = (
    'max_episodes_rendered = '
    'int(os.environ.get("LEROBOT_MAX_RECORDED", "0"))'
)
if _flexible_patch not in _src:
    if _broken_patch in _src:
        # 過去に broken 版が入っていたら修正
        _src = _src.replace(_broken_patch, _flexible_patch, 1)
    else:
        _old_patch = "max_episodes_rendered = 0"
        if _old_patch not in _src:
            raise RuntimeError(
                "eval_script のパッチ地点が見つかりません。"
                "Section 5 を先に実行してください。"
            )
        _src = _src.replace(_old_patch, _flexible_patch, 1)
    eval_script_path.write_text(_src, encoding="utf-8")

# --- 2. 録画付き eval を実行 ------------------------------------
shutil.rmtree(VIDEO_OUT_DIR, ignore_errors=True)

video_env = os.environ.copy()
video_env["MUJOCO_GL"] = "egl"
video_env["PYTHONPATH"] = (
    str(LIBERO_PLUS_DIR)
    + os.pathsep
    + str(LEROBOT_SRC)
    + os.pathsep
    + video_env.get("PYTHONPATH", "")
)
video_env["LEROBOT_MAX_RECORDED"] = str(VIDEO_N_EPISODES)
video_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
video_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
video_env["HF_HUB_VERBOSITY"] = "error"
video_env["TQDM_DISABLE"] = "1"
video_env["PYTHONWARNINGS"] = "ignore"
video_env["PYTHONUNBUFFERED"] = "1"

video_command = [
    "lerobot-eval",
    f"--policy.path={MERGED_MODEL_DIR}",
    "--policy.device=cuda",
    "--policy.use_amp=false",
    "--env.type=libero",
    "--env.is_libero_plus=true",
    "--env.task=libero_spatial",
    "--env.task_ids="
    + json.dumps([VIDEO_TASK_ID], separators=(",", ":")),
    "--env.camera_name_mapping="
    + json.dumps(EVAL_CAMERA_MAPPING, separators=(",", ":")),
    "--env.observation_height=256",
    "--env.observation_width=256",
    "--env.control_mode=relative",
    "--env.max_parallel_tasks=1",
    "--eval.batch_size=1",
    f"--eval.n_episodes={VIDEO_N_EPISODES}",
    "--eval.use_async_envs=false",
    # recording=false + LEROBOT_MAX_RECORDED>0 の組み合わせで、
    # eval_script が videos_dir に mp4 を吐き出すルートに乗る (recording=true は
    # HF dataset 形式の完全記録用で mp4 とは排他).
    "--eval.recording=false",
    f"--seed={EVAL_SEED}",
    f"--output_dir={VIDEO_OUT_DIR}",
]

print(
    f"Recording {VIDEO_N_EPISODES} rollout(s) on task {VIDEO_TASK_ID}...",
    flush=True,
)
_video_process = subprocess.Popen(
    video_command,
    cwd=LEROBOT_DIR,
    env=video_env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert _video_process.stdout is not None
_video_recent: deque[str] = deque(maxlen=120)
_prog_pat = re.compile(
    r"^EVAL_PROGRESS "
    r"task=(\d+)/(\d+) "
    r"episode=(\d+)/(\d+)$"
)
for _raw in _video_process.stdout:
    _line = _raw.replace("\r", "").strip()
    if not _line:
        continue
    _video_recent.append(_line)
    _m = _prog_pat.match(_line)
    if _m:
        _, _, ei, et = _m.groups()
        print(f"  episode {ei}/{et}", flush=True)
_rc = _video_process.wait()
if _rc != 0:
    print("\n".join(_video_recent))
    raise RuntimeError(f"video eval failed: {_rc}")

# --- 3. できた mp4 を notebook に埋め込み ------------------------
mp4_files = sorted(VIDEO_OUT_DIR.rglob("*.mp4"))
if not mp4_files:
    raise RuntimeError(
        f"mp4 が {VIDEO_OUT_DIR} に見つかりません。"
    )

print(f"\nFound {len(mp4_files)} mp4 file(s):")
for _mp4 in mp4_files:
    print(f"  {_mp4.relative_to(VIDEO_OUT_DIR)}")
    # embed=True にすると base64 で埋め込まれ notebook 単体で再生できる
    # (ただしファイル増える)。単に参照だけしたければ embed=False に。
    display(Video(str(_mp4), embed=True, html_attributes="controls loop"))

## 9. アドバンスド評価: LIBERO-plus 4 suite × 3 タスク × 1 エピソード

Section 8 で **自分なりの工夫を加えた LeRobot 版モデル** を、この Section 9 で
**より広い suite** (Spatial 以外も含む) で評価します。

工夫のやり方や注意点は **Section 8.0 の「工夫できる箇所」** を参照。
ここでは Section 9 側の **入力形式** と **出力形式** だけを説明します。

---

### 📥 入力形式: Section 9 が要求するモデルディレクトリ

Section 9 は 2 つのディレクトリを `lerobot-eval` に渡します:

- `BASELINE_MODEL_DIR` = Section 8.4 で作った **対照ベースライン** (公開重み)
- `MERGED_MODEL_DIR`   = Section 8.3 で作った **あなたが追加学習したモデル**

どちらも以下のファイル構成でなければ `lerobot-eval` がロードできません:

必要なファイル (`{POLICY_DIR}/` 直下):

- `model.safetensors` — 本体の重み (必須)
- `config.json` — モデル config (必須)
- `policy_preprocessor.json` + `policy_preprocessor*.safetensors` — 入力前処理設定 + 統計量
- `policy_postprocessor.json` + `policy_postprocessor*.safetensors` — 出力後処理設定 + 統計量

**この形式は `SmolVLAPolicy.save_pretrained(POLICY_DIR)` で自動的に作られます**。
Section 8.3 (`f1e24ae5` セル) と 8.4 (`ce8fe958` セル) は既にこれを呼んでいるので、
**そこを触らない限り自動で守られます**。

Section 9 の冒頭にある **preflight check セル** が、必要ファイルが揃っているかを事前に確認します。

---

### 🔬 出力形式: このセクションが吐く成果物

`workdir/eval/advanced/` 以下に、モデル × suite ごとの LeRobot 標準評価出力:

- `baseline/{libero_spatial,libero_object,libero_goal,libero_10}/eval_info.json` — 対照ベースライン (各 suite 分)
- `finetuned/{libero_spatial,libero_object,libero_goal,libero_10}/eval_info.json` — LoRA 学習後モデル分
- `advanced_comparison.csv` — セル末尾で作る集計表

`eval_info.json` の中身 (LeRobot v0.6.0 のフォーマット):

```json
{
  "overall": {
    "pc_success": 65.0,          // suite 全体の成功率 (%)
    "avg_reward": ...,
    "avg_sum_reward": ...
  },
  "per_task": [
    {
      "task_id": 0,
      "task_group": "libero_spatial",
      "metrics": {
        "successes": [true],  // 各 episode の成否 (長さ = 1)
        "rewards": [...],
        ...
      }
    }
    // ...
  ]
}
```

セル末尾で pandas DataFrame にまとめ、集計表を出力・CSV 保存します。 表の列は
`Suite / Tasks / Base (%) / Spatial LoRA (%) / Δ (pp)` で、各 suite 5 タスクの
結果と全体平均 (Overall = 4 suite × 3 task × 1 ep = 12 rollout) を並べます。

**モデルあたり 12 rollout × 2 モデル = 計 24 rollout。**
評価は多様性優先で 1 ep のみとしています (成功率は少数試行なのでノイズ大。
最終的な結論を出すには本来はもっとエピソードが必要な点に注意)。


In [ ]:
# ==============================================================
# 9.0 Preflight check: section 9 が要求する入力形式を満たしているか
# ==============================================================
# section 8 で自分なりの工夫をした結果、モデルディレクトリの形式が
# 壊れていないか事前に確認する. ここで失敗するなら section 8 を見直す.

_REQUIRED_FILES = [
    "model.safetensors",
    "config.json",
    "policy_preprocessor.json",
    "policy_postprocessor.json",
]
# .safetensors の stats は名前が可変なので glob で検証
_REQUIRED_GLOBS = [
    "policy_preprocessor*.safetensors",
    "policy_postprocessor*.safetensors",
]

def _check_policy_dir(policy_dir: Path, label: str) -> None:
    print(f"\n[{label}] {policy_dir}")
    if not policy_dir.is_dir():
        raise FileNotFoundError(
            f"{label}: ディレクトリが存在しません。"
            "section 8 の該当セルを実行しましたか?"
        )
    missing = []
    for name in _REQUIRED_FILES:
        p = policy_dir / name
        ok = p.is_file()
        print(f"  {'✅' if ok else '❌'}  {name}"
              + (f"  ({p.stat().st_size / 1e6:.1f} MB)" if ok else ""))
        if not ok:
            missing.append(name)
    for pattern in _REQUIRED_GLOBS:
        matches = list(policy_dir.glob(pattern))
        ok = len(matches) > 0
        print(f"  {'✅' if ok else '❌'}  {pattern}  ({len(matches)} match)")
        if not ok:
            missing.append(pattern)
    if missing:
        raise FileNotFoundError(
            f"{label} に必要ファイルが不足: {missing}. "
            "section 8.3 (merge) / 8.4 (baseline) を再実行してください。"
        )


_check_policy_dir(BASELINE_MODEL_DIR, "BASELINE_MODEL_DIR (対照)")
_check_policy_dir(MERGED_MODEL_DIR, "MERGED_MODEL_DIR (工夫版)")
print("\n✅ 契約 OK — section 9 のアドバンスド評価を実行できます")

In [ ]:
# ==============================================================
# アドバンスド評価: 4 suite × 3 task × 1 ep = 12 rollout / モデル
# ==============================================================
# タスクは各 suite の 10 タスクから等間隔に 3 タスクを取り、
# 「同じ task を繰り返す」より「異なる task を広く回す」方針で多様性を優先.
# 1 モデルあたり 12 rollout, 2 モデル (base + finetuned) で計 24 rollout.
#
# 内訳:
#   libero_spatial : 3 tasks (学習ドメイン)
#   libero_object  : 3 tasks (オブジェクト汎化)
#   libero_goal    : 3 tasks (目標語彙汎化)
#   libero_10      : 3 tasks (long-horizon 汎化)

# 各 suite の 10 tasks (id 0-9) から等間隔に 3 個をサンプル (start / middle / end)
_ADV_TASK_IDS = [0, 4, 8]

ADVANCED_SUITES = {
    "libero_spatial": _ADV_TASK_IDS,
    "libero_object":  _ADV_TASK_IDS,
    "libero_goal":    _ADV_TASK_IDS,
    "libero_10":      _ADV_TASK_IDS,
}
ADVANCED_EPISODES_PER_TASK = 1
ADVANCED_OUT_DIR = WORKDIR / "eval" / "advanced"


def run_advanced_eval(policy_path, out_root, label):
    """1 モデルについて全 suite を回して suite→eval_info の dict を返す。"""
    shutil.rmtree(out_root, ignore_errors=True)
    suite_results = {}
    for suite_name, task_ids in ADVANCED_SUITES.items():
        sub_dir = out_root / suite_name

        adv_env = os.environ.copy()
        adv_env["MUJOCO_GL"] = "egl"
        adv_env["PYTHONPATH"] = (
            str(LIBERO_PLUS_DIR)
            + os.pathsep
            + str(LEROBOT_SRC)
            + os.pathsep
            + adv_env.get("PYTHONPATH", "")
        )
        adv_env["LEROBOT_MAX_RECORDED"] = "0"
        adv_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        adv_env["TQDM_DISABLE"] = "1"
        adv_env["PYTHONWARNINGS"] = "ignore"
        adv_env["PYTHONUNBUFFERED"] = "1"

        adv_command = [
            "lerobot-eval",
            f"--policy.path={policy_path}",
            "--policy.device=cuda",
            "--policy.use_amp=false",
            "--env.type=libero",
            "--env.is_libero_plus=true",
            f"--env.task={suite_name}",
            "--env.task_ids="
            + json.dumps(task_ids, separators=(",", ":")),
            "--env.camera_name_mapping="
            + json.dumps(
                EVAL_CAMERA_MAPPING, separators=(",", ":")
            ),
            "--env.observation_height=256",
            "--env.observation_width=256",
            "--env.control_mode=relative",
            "--env.max_parallel_tasks=1",
            "--eval.batch_size=1",
            f"--eval.n_episodes={ADVANCED_EPISODES_PER_TASK}",
            "--eval.use_async_envs=false",
            "--eval.recording=false",
            f"--seed={EVAL_SEED}",
            f"--output_dir={sub_dir}",
        ]
        print(
            f"\n[{label}] {suite_name} "
            f"({len(task_ids)} tasks × "
            f"{ADVANCED_EPISODES_PER_TASK} ep)",
            flush=True,
        )
        proc = subprocess.Popen(
            adv_command,
            cwd=LEROBOT_DIR,
            env=adv_env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        assert proc.stdout is not None
        adv_recent: deque[str] = deque(maxlen=120)
        _prog = re.compile(
            r"^EVAL_PROGRESS "
            r"task=(\d+)/(\d+) "
            r"episode=(\d+)/(\d+)$"
        )

        # tqdm でこの suite の rollout 進捗を出す
        from tqdm.auto import tqdm
        _total_this_suite = len(task_ids) * ADVANCED_EPISODES_PER_TASK
        _pbar_adv = tqdm(
            total=_total_this_suite,
            desc=f"{label} {suite_name}",
            unit="ep",
        )

        for raw in proc.stdout:
            line = raw.replace("\r", "").strip()
            if not line:
                continue
            adv_recent.append(line)
            m = _prog.match(line)
            if m:
                ti, tt, ei, et = m.groups()
                _pbar_adv.update(1)
                _pbar_adv.set_postfix_str(f"task {ti}/{tt} ep {ei}/{et}")

        _pbar_adv.close()
        rc = proc.wait()
        if rc != 0:
            raise RuntimeError("\n".join(adv_recent))
        info = json.loads(
            (sub_dir / "eval_info.json").read_text(
                encoding="utf-8"
            )
        )
        suite_results[suite_name] = info
    return suite_results


print("=== Advanced eval: baseline ===")
adv_base = run_advanced_eval(
    BASELINE_MODEL_DIR,
    ADVANCED_OUT_DIR / "baseline",
    "Base",
)

print("\n=== Advanced eval: fine-tuned ===")
adv_ft = run_advanced_eval(
    MERGED_MODEL_DIR,
    ADVANCED_OUT_DIR / "finetuned",
    "LoRA",
)


# --- 全 rollout 合計での成功率 ---------------------------------
def overall_across_suites(results):
    total_success = 0.0
    total_rollouts = 0
    for info in results.values():
        for task in info["per_task"]:
            successes = task["metrics"]["successes"]
            total_success += sum(bool(v) for v in successes)
            total_rollouts += len(successes)
    return (
        100.0 * total_success / total_rollouts
        if total_rollouts
        else 0.0
    )


# --- suite ごとの集計テーブル ----------------------------------
adv_rows = []
for suite_name in ADVANCED_SUITES:
    b = adv_base[suite_name]["overall"]["pc_success"]
    f = adv_ft[suite_name]["overall"]["pc_success"]
    adv_rows.append({
        "Suite": suite_name,
        "Tasks": len(ADVANCED_SUITES[suite_name]),
        "Base (%)": b,
        "Spatial LoRA (%)": f,
        "Δ (pp)": f - b,
    })

adv_base_overall = overall_across_suites(adv_base)
adv_ft_overall = overall_across_suites(adv_ft)
adv_rows.append({
    "Suite": "Overall (12 rollouts)",
    "Tasks": sum(len(v) for v in ADVANCED_SUITES.values()),
    "Base (%)": adv_base_overall,
    "Spatial LoRA (%)": adv_ft_overall,
    "Δ (pp)": adv_ft_overall - adv_base_overall,
})

adv_df = pd.DataFrame(adv_rows)
adv_csv = ADVANCED_OUT_DIR / "advanced_comparison.csv"
adv_df.to_csv(adv_csv, index=False)
display(adv_df.round(1))
print(f"\nSaved: {adv_csv}")
print(
    f"Overall (12 rollouts): {adv_base_overall:.1f}% -> "
    f"{adv_ft_overall:.1f}% "
    f"({adv_ft_overall - adv_base_overall:+.1f} pp)"
)